<a href="https://colab.research.google.com/github/MuthoniThuo/unsupervised-learning/blob/main/Kmeans_example_unsupervised_learning_lab_work%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sections 1
# What each import does:
# Generate Sample Data
Libraries used:

NumPy handles all mathematical operations and array calculations.

Matplotlib creates all the visualizations and plots.
make_blobs is a tool that generates synthetic datasets with distinct groups of points.

**Data generation:**
The function creates 300 data points arranged into 3 separate clusters. Each cluster has a controlled spread so the groups are visible but not perfectly separated. A random seed is set so the exact same data appears every time the code runs – this ensures all students see identical results.

# The outputs:
X contains the actual coordinates of every point (each point has an x and y value). y_true stores the correct group label for each point – this is the answer key used later to check accuracy.

# The plot:
A scatter plot draws every point on a 2D graph. The x-axis is Feature 1, the y-axis is Feature 2. The title asks students if they can spot the groups visually before the algorithm runs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generate 3 clusters of points
X, y_true = make_blobs(
    n_samples=300,      # 300 points
    centers=3,          # 3 true clusters
    cluster_std=1.0,    # spread of each cluster
    random_state=42
)

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.7)
plt.title("Unlabeled Data – Can you see the groups?")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, alpha=0.3)
plt.show()

# section 2
This function takes three inputs: the dataset, the number of clusters K, and a maximum number of iterations to prevent infinite looping.

**Step 1** – Random Initialization:
K random data points are selected as the starting centroids. No point can be picked twice. These are the initial "tables" in the cafeteria analogy.

**Step 2** – Main Loop (repeats until convergence):

**Step 2a – Compute Distances:**
For every data point, the straight-line (Euclidean) distance to each centroid is calculated. A grid is created where each row is a point and each column is a centroid, with distances filling the cells.

**Step 2b – Assign Labels:**
Each point is assigned to whichever centroid is closest. The result is a label (0, 1, or 2) for every point. This is the "students sit at the nearest table" step.

**Step 2c – Update Centroids:**
Each centroid moves to the average position of all points assigned to it. If a cluster has no points, the centroid stays where it is to avoid errors.

**Step 2d – Check Convergence:**
If the new centroids are essentially in the same positions as the old ones, the algorithm stops early. Otherwise, centroids are updated and the loop continues.

**Saving history:**
At each iteration, the current centroids and labels are saved into a list. This allows replaying the entire process later as an animation.

# The final output:
The function returns three things – the final centroid positions, the final cluster labels for every point, and the full history for animation.

**The comparison plot:**
Two graphs are shown side by side. The left graph colors points by their true cluster labels. The right graph colors points by what K-Means predicted. The centroids are drawn as large red X's. Students can visually compare how well the algorithm matched reality.

In [ ]:
def kmeans_scratch(X, K, max_iters=10):
    """
    K-Means from scratch – step by step

    Parameters:
    X : array of data points
    K : number of clusters
    max_iters : maximum iterations

    Returns:
    centroids, labels, history (for animation)
    """

    # STEP 1: Randomly initialize K centroids
    # Pick K random points from the data
    random_indices = np.random.choice(len(X), K, replace=False)
    centroids = X[random_indices].copy()

    history = []  # Store centroids at each step for visualization

    for iteration in range(max_iters):
        # STEP 2: Assign each point to the NEAREST centroid
        # Compute Euclidean distance from each point to each centroid
        distances = np.zeros((len(X), K))
        for k in range(K):
            distances[:, k] = np.sqrt(np.sum((X - centroids[k])**2, axis=1))

        # Get the cluster label (0, 1, or 2) for each point
        labels = np.argmin(distances, axis=1)

        # Save state for animation
        history.append((centroids.copy(), labels.copy()))

        # STEP 3: Update centroids – move to center of assigned points
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            # Get all points belonging to cluster k
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                # New centroid = mean of all points in the cluster
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                # If a cluster gets empty, keep the old centroid
                new_centroids[k] = centroids[k]

        # STEP 4: Check for convergence
        # If centroids didn't move, we're done!
        if np.allclose(centroids, new_centroids):
            print(f"✓ Converged after {iteration + 1} iterations!")
            break

        centroids = new_centroids

    return centroids, labels, history


# Run our K-Means
K = 3
final_centroids, final_labels, history = kmeans_scratch(X, K)

# Visualize the result
plt.figure(figsize=(10, 5))

# Left: True clusters
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title("True Clusters (Ground Truth)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

# Right: K-Means result
plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=final_labels, cmap='viridis', s=50, alpha=0.7)
plt.scatter(final_centroids[:, 0], final_centroids[:, 1],
            c='red', s=200, marker='X', label='Centroids', edgecolors='black')
plt.title(f"K-Means Result (K={K})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.tight_layout()
plt.show()

# Animation of Converging Centroids
This section creates a moving visualization showing how clusters form over time.

A function is set up to redraw the plot for each iteration. At every frame:

**The plot is wiped clean.**

The centroids and labels from that specific iteration are loaded from saved history.

Points are colored by their current cluster assignment.
Centroids are drawn as large red X's.

The title updates to show which iteration number is displayed.

The animation runs with a 700-millisecond delay between frames and loops continuously.
 Students watch as centroids bounce around and then settle into their final positions.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

def animate(iteration):
    ax.clear()
    centroids, labels = history[iteration]

    # Plot points colored by cluster
    for k in range(K):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[k],
                   s=50, alpha=0.6, label=f'Cluster {k}')

    # Plot centroids
    ax.scatter(centroids[:, 0], centroids[:, 1],
               c='red', s=250, marker='X',
               edgecolors='black', linewidth=2, label='Centroids')

    ax.set_title(f'K-Means Iteration {iteration + 1}/{len(history)}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)

ani = FuncAnimation(fig, animate, frames=len(history), interval=700, repeat=True)
plt.close()
HTML(ani.to_jshtml())  # This creates the interactive animation

# The Elbow Method
**Inertia calculation:**

A function measures how "tight" clusters are by summing the squared distances from every point to its assigned centroid. Lower inertia means points are packed closer together.

**Testing different K values: **

K-Means runs for K values from 1 to 9. For each value, the inertia is calculated and stored.

**The elbow plot:**
A line graph is drawn with K on the x-axis and inertia on the y-axis. As K increases, inertia always decreases (more centroids mean points are closer to one of them).

But at the optimal K, the drop in inertia slows dramatically, forming an "elbow" shape. A vertical red dashed line marks where K=3, the true number of clusters.

The key insight is: adding clusters beyond the elbow gives diminishing returns in tightness.

In [ ]:
def compute_inertia(X, labels, centroids, K):
    inertia = 0
    for k in range(K):
        cluster_points = X[labels == k]
        if len(cluster_points) > 0:
            # Sum of squared distances to centroid
            inertia += np.sum((cluster_points - centroids[k])**2)
    return inertia

# Test different K values
K_range = range(1, 10)
inertias = []

for K in K_range:
    centroids, labels, _ = kmeans_scratch(X, K)
    inertias.append(compute_inertia(X, labels, centroids, K))

# Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='True K = 3')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method – Find the Best K\n(Look for the "bend" in the curve)')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Real-World Customer Segmentation
# Pandas:
A library that adds spreadsheet-like functionality for organizing and analyzing data.

Simulating customer data:
200 fictitious customers are generated across three hidden segments:


70 budget customers earning around $35,000 with low spending

70 middle-class customers earning around $60,000 with moderate spending

60 premium customers earning around $95,000 with high spending

Each group has realistic variation around its averages.

**Scaling the data:**

A standard scaler transforms the data so all features have equal importance. Without this, Income (ranging from $20k to $120k) would completely dominate Spending Score (ranging from 0 to 100) in distance calculations. After scaling, both features contribute equally.

**Running K-Means properly:**

Scikit-learn's optimized K-Means is used with K=3. It runs 10 times with different random starts and keeps the best result – this solves the initialization sensitivity problem.

**Visualizing results:**

Points are colored by their assigned cluster. Centroids are converted back to original units (actual dollars and spending scores) so axes show meaningful numbers.

**Interpreting segments:**

The code prints the average income and average spending score for each discovered cluster. This helps students give real-world meaning to each group – for example, "Cluster 0 is young budget-conscious shoppers" or "Cluster 2 is affluent high-spenders."

In [ ]:
import pandas as pd

# Simulate customer data
np.random.seed(42)
n_customers = 200

# Generate realistic customer segments
data = {
    'Annual_Income': np.concatenate([
        np.random.normal(35000, 5000, 70),   # Budget customers
        np.random.normal(60000, 7000, 70),    # Middle class
        np.random.normal(95000, 10000, 60)    # Premium customers
    ]),
    'Spending_Score': np.concatenate([
        np.random.normal(30, 10, 70),    # Low spenders
        np.random.normal(60, 10, 70),    # Medium spenders
        np.random.normal(85, 8, 60)      # High spenders
    ])
}

df = pd.DataFrame(data)

# Normalize the data (important for K-Means!)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Apply K-Means
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df['Annual_Income'], df['Spending_Score'],
                      c=df['Cluster'], cmap='Set2', s=100, alpha=0.8)

# Add centroids (in original scale)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            c='red', s=300, marker='X', edgecolors='black', linewidth=2)

plt.xlabel('Annual Income ($)')
plt.ylabel('Spending Score (0-100)')
plt.title('Customer Segmentation using K-Means')
plt.colorbar(scatter, label='Segment')

# Add customer profiles
for i, row in df.iterrows():
    if i % 20 == 0:  # Label every 20th point
        plt.annotate(f'Customer {i}',
                    (row['Annual_Income'], row['Spending_Score']),
                    fontsize=8, alpha=0.7)

plt.grid(True, alpha=0.3)
plt.show()

# Print segment profiles
print("\n📊 Customer Segment Profiles:\n")
for cluster in sorted(df['Cluster'].unique()):
    segment = df[df['Cluster'] == cluster]
    print(f"--- Segment {cluster} ---")
    print(f"  Customers: {len(segment)}")
    print(f"  Avg Income:  ${segment['Annual_Income'].mean():.0f}")
    print(f"  Avg Spending: {segment['Spending_Score'].mean():.1f}/100")
    print()